# 第12章　不確実性と棄却を実装する ― AIに「わかりません」と言わせる

**『本格実装 医療診断支援AI（実装編）』のコード**

本文に載っているコードを、章の順にそのまま収めています。紙面のコードは読んで理解するためのもの、こちらは動かすためのものです。

- Python 以外（シェル・YAML・Dockerfile など）は、実行環境が違うので**コードセルにせず、そのまま読める形で置いています**。使う場所を確かめてから実行してください。
- 抜粋である以上、上から順に実行するだけで通るとは限りません。データの取得先やパスは、お手元の環境に合わせてください。
- **教育・研究のためのコードです。患者データをこのノートブックに置かないでください。**

リポジトリ: https://github.com/kewel-corp/book-impl

## 12.1　確信度で、棄却する

In [ ]:
import torch

@torch.no_grad()
def predict_with_abstain(model, x, low=0.6):
    prob = model(x).softmax(1)
    conf, pred = prob.max(1)
    result = {"pred": pred.item(), "conf": conf.item()}
    if conf.item() < low:                        # 自信がなければ棄権
        result["flag"] = "要専門医確認"
    return result

## 12.2　モデルの不一致で、不確実性を測る

In [ ]:
import torch

@torch.no_grad()
def ensemble_uncertainty(models, x):
    probs = torch.stack([m(x).softmax(1) for m in models])   # (M, B, C)
    mean = probs.mean(0)                                      # 平均予測
    disagreement = probs.std(0).mean(dim=1)                  # 症例ごとのばらつき (B,)
    return {"pred": mean.argmax(1),                           # (B,) のまま返す（バッチ推論に対応）
            "uncertainty": disagreement}                     # (B,) 大きいほど不確実

## 12.3　MCドロップアウト ― 一つのモデルで揺らぎを見る

In [ ]:
import torch

def enable_dropout(model):
    """Dropout だけを train に戻す。model.train() だとBatchNormまで
    学習モードになり、running統計が推論のたびに書き換わってしまう。"""
    for m in model.modules():
        if m.__class__.__name__.startswith("Dropout"):
            m.train()                            # 推論時もドロップアウトを効かせる

def snapshot_modes(model):
    """module ごとの training フラグを控える（モデル全体の1フラグでは、層ごとに混在した状態を戻せない）"""
    return [(m, m.training) for m in model.modules()]

def restore_modes(snapshot):
    # modules() は親→子の順なので、親の train()/eval() が子へ伝播しても、後から子が個別に上書きされる
    for m, flag in snapshot:
        m.train(flag)

@torch.no_grad()
def mc_dropout(model, x, n=20):
    saved = snapshot_modes(model)                 # 呼ぶ前の状態を module 単位で控える
    try:
        model.eval()                              # まず全体を評価モードへ（BatchNorm は貯めた統計を使う）
        enable_dropout(model)                     # そのうえでDropoutだけ有効にする
        # eval() を挟まないと、元がtrainのモデルでは BatchNorm もバッチ統計で動いてしまい、
        # 「ドロップアウトによる揺らぎ」に別の要因が混ざる。
        probs = torch.stack([model(x).softmax(1) for _ in range(n)])  # (n, B, C)
    finally:
        restore_modes(saved)                      # 例外が出ても、必ず呼ぶ前の状態へ戻す
    mean = probs.mean(0)                                          # (B, C)
    # std().mean() で全症例を1つの数字に潰さない。不確かさは症例ごとに返す。
    std_per_case = probs.std(0, unbiased=(n > 1)).mean(1)         # (B,)
    return mean, std_per_case

## 手を動かす ― 確信度を較正してから、棄却率を選ぶ

In [ ]:
import torch
def fit_temperature(logits_val, y_val, device="cpu"):
    # 検証のlogitsは計算グラフから切り離し、deviceを揃える（LBFGSはclosureを何度も呼ぶ）
    logits_val = logits_val.detach().to(device)
    y_val      = y_val.detach().to(device)
    # T を直接最適化すると負値やゼロになりうる。log_T を最適化して T=exp(log_T)>0 を保証する
    log_T = torch.zeros(1, device=device, requires_grad=True)
    opt = torch.optim.LBFGS([log_T], lr=0.01, max_iter=50)
    nll = torch.nn.CrossEntropyLoss()
    def closure():
        opt.zero_grad()
        loss = nll(logits_val / log_T.exp(), y_val)   # T>1 なら自信を下げる方向
        loss.backward()
        return loss
    opt.step(closure)
    return float(log_T.detach().exp())

T = fit_temperature(logits_val, y_val)
prob = (logits_test / T).softmax(1)           # 較正済み確率で棄却を判断
# 適合させた分布（どの施設・どの装置の検証データか）とTを、モデルと一緒に保存する。
# NLLを下げてもECEが必ず縮むとは限らないので、較正は独立したデータで評価する。

## 予測集合の被覆率を保証する ― コンフォーマル予測

In [ ]:
import numpy as np
def conformal_qhat(prob_cal, y_cal, alpha=0.1):
    prob_cal, y_cal = np.asarray(prob_cal, dtype=float), np.asarray(y_cal, dtype=int)
    n = len(y_cal)
    assert n > 0 and 0 < alpha < 1 and np.isfinite(prob_cal).all(), "入力が不正"
    s = np.sort(1 - prob_cal[np.arange(n), y_cal])        # 較正データの非適合スコア（昇順）
    k = int(np.ceil((n + 1) * (1 - alpha)))               # k 番目の順序統計量を閾値にする
    if k > n:                                             # 較正例が少なすぎる：閾値∞＝全クラスを返す（何も絞れない）
        return np.inf
    return float(s[k - 1])

def predict_set(prob, qhat):
    return np.where(1 - prob <= qhat)[0]                  # 予測集合（クラスの集合）